In [ ]:
import re
import nltk
import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem.snowball import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
df = pd.read_csv('VKM_dataset_cleaned.csv')

# Combineer relevante tekstkolommen
text_cols = ['name','shortdescription','description','content','learningoutcomes','module_tags']
for col in text_cols:
    df[col] = df[col].fillna('')

df['combined_text'] = df[text_cols].agg(' '.join, axis=1)

In [ ]:
stop_words = set(stopwords.words('dutch'))
stemmer = SnowballStemmer('dutch')

def preprocess(text: str) -> str:
    t = text.lower()
    t = re.sub(r'\d+', ' ', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    tokens = []
    for sentence in sent_tokenize(t):
        words = word_tokenize(sentence)
        words = [w for w in words if w not in stop_words and len(w) > 2]
        words = [stemmer.stem(w) for w in words]
        tokens.extend(words)
    return ' '.join(tokens)

df['text_clean'] = df['combined_text'].apply(preprocess)

In [ ]:
vectorizer = TfidfVectorizer(max_df=0.85, min_df=2, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(df['text_clean'])
feature_names = vectorizer.get_feature_names_out()

In [ ]:
def build_profile_text(interests, values, goals, free_text=''):
    combined = ' '.join(interests + values + goals + [free_text])
    return preprocess(combined)

# Voorbeeldprofiel A: Tech & Innovatie
profile_text_A = build_profile_text(
    interests=['technologie','zorg','robotica','ai'],
    values=['duurzaamheid','impact'],
    goals=['innovatie','praktijkproject'],
    free_text='ik wil leren bouwen en testen van prototypes die echte waarde leveren in de zorg.'
)

student_vec_A = vectorizer.transform([profile_text_A])
similarities_A = cosine_similarity(student_vec_A, tfidf_matrix)[0]
df['cosine_sim'] = similarities_A

In [ ]:
scaler = MinMaxScaler()
df[['cosine_sim','interests_match_score','popularity_score']] = scaler.fit_transform(
    df[['cosine_sim','interests_match_score','popularity_score']]
)

df['hybrid_score'] = (
    0.6 * df['cosine_sim'] +
    0.25 * df['interests_match_score'] +
    0.15 * df['popularity_score']
)

top5_A = df.sort_values('hybrid_score', ascending=False).head(5)
print(top5_A[['name','location','studycredit','hybrid_score']])

                                                  name            location  \
17   technologie die ècht werkt: innovatie in zorg ...               Breda   
13                                       zorg dichtbij               Breda   
12                      technologie in zorg en welzijn           Den Bosch   
43                                    palliatieve zorg               Breda   
191                      innovatie door service design  Breda en Den Bosch   

     studycredit  hybrid_score  
17            15      0.855261  
13            15      0.729167  
12            30      0.642885  
43            30      0.566016  
191           30      0.523919  


In [ ]:
def explain(row, student_tokens, top_features=5):
    idx = row.name
    doc_vec = tfidf_matrix[idx].toarray()[0]
    top_idx = np.argsort(doc_vec)[::-1][:top_features]
    top_terms = [feature_names[i] for i in top_idx]
    overlap = sorted(list(set(student_tokens) & set(row['module_tags'].lower().split(', '))))
    return {'matched_tags': overlap, 'top_terms': top_terms}

student_tokens_A = set(profile_text_A.split())
for _, row in top5_A.iterrows():
    exp = explain(row, student_tokens_A)
    print(f"Module: {row['name']}")
    print("Matched tags:", exp['matched_tags'])
    print("Top TF-IDF terms:", exp['top_terms'])
    print()

Module: technologie die ècht werkt: innovatie in zorg en welzijn
Matched tags: ['robotica', 'technologie', 'zorg']
Top TF-IDF terms: ['zorg', 'technologie', 'echt', 'zorg welzijn', 'welzijn']

Module: zorg dichtbij
Matched tags: ['technologie', 'zorg']
Top TF-IDF terms: ['zorg', 'technologie', 'implementer', 'praktijkgericht modul', 'thuis']

Module: technologie in zorg en welzijn
Matched tags: ['technologie', 'zorg']
Top TF-IDF terms: ['technologie', 'zorg welzijn', 'welzijn', 'technologie zorg', 'lat']

Module: palliatieve zorg
Matched tags: ['zorg']
Top TF-IDF terms: ['palliatiev zorg', 'palliatiev', 'zorg', 'aanvull', 'zorg verdiep']

Module: innovatie door service design
Matched tags: ['innovatie', 'zorg']
Top TF-IDF terms: ['zorg', 'design', 'verdiep', 'innovatie', 'del']



In [ ]:
profile_text_B = build_profile_text(
    interests=['jeugdzorg','psychologie','ontwikkeling','trauma'],
    values=['communicatie','mensgericht'],
    goals=['gespreksvoering','reflectie'],
    free_text='ik wil leren begeleiden en analyseren in complexe situaties met aandacht voor rouw en verlies.'
)

student_vec_B = vectorizer.transform([profile_text_B])
similarities_B = cosine_similarity(student_vec_B, tfidf_matrix)[0]
df['cosine_sim'] = similarities_B

df[['cosine_sim','interests_match_score','popularity_score']] = scaler.fit_transform(
    df[['cosine_sim','interests_match_score','popularity_score']]
)

df['hybrid_score'] = (
    0.6 * df['cosine_sim'] +
    0.25 * df['interests_match_score'] +
    0.15 * df['popularity_score']
)

top5_B = df.sort_values('hybrid_score', ascending=False).head(5)
print(top5_B[['name','location','studycredit','hybrid_score']])

                                                  name            location  \
25   going global: internationaal perspectief op je...  Breda en Den Bosch   
46   patronen doorbreken / sociale verbinding & wel...               Breda   
18                          patronen doorbreken: basis  Breda en Den Bosch   
0                         kennismaking met psychologie           Den Bosch   
170  minor forensisch onderzoek in de rechtbank- (i...  Breda en Den Bosch   

     studycredit  hybrid_score  
25            15      0.766780  
46            30      0.758080  
18            15      0.748253  
0             15      0.663098  
170           30      0.652673  
